In [ ]:
# !pip install -q -U google-genai yfinance duckduckgo-search pydantic
# !pip install ddgs
# !pip install -q -U ddgs yfinance google-genai pydantic
# !pip install -q -U streamlit

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
import yfinance as yf
import json

from ddgs import DDGS

import time
from google.genai.errors import APIError

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Initialize the official Google GenAI Client
client = genai.Client()

In [ ]:
def safe_generate_content(prompt: str, primary_model: str = "gemini-3.5-flash", fallback_model: str = "gemini-3.1-flash-lite", max_retries: int = 3) -> str:
    """
    Calls the Gemini API with automatic exponential backoff and model fallback.
    """
    base_delay = 2  # Start by waiting 2 seconds
    
    # Attempt with Primary Model
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=primary_model,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.2)
            )
            return response.text
            
        except Exception as e:
            # Check if it's a 503 Overloaded error or 429 Rate Limit
            error_str = str(e)
            if "503" in error_str or "UNAVAILABLE" in error_str or "429" in error_str:
                wait_time = base_delay * (2 ** attempt)
                print(f"  [Warning] Server busy ({primary_model}). Retrying in {wait_time}s... (Attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                # If it's a different error (e.g., invalid API key), raise it immediately
                print(f"  [Error] Unhandled API exception: {e}")
                break
                
    # If primary model fails after all retries, switch to Fallback Model
    print(f"  [Fallback] Primary model '{primary_model}' unreachable. Switching to backup '{fallback_model}'...")
    try:
        response = client.models.generate_content(
            model=fallback_model,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.2)
        )
        return response.text + "\n\n*(Note: Generated using backup model due to high server demand)*"
    except Exception as fallback_error:
        return f"CRITICAL ERROR: Both primary and fallback models failed. Details: {str(fallback_error)}"

In [ ]:

def get_recent_news(ticker: str, max_results: int = 5) -> str:
    """
    Fetches news using DDGS first. 
    If DDGS fails or returns empty, automatically falls back to Yahoo Finance news.
    """
    results = []
    
    # Attempt 1: Primary Tool (DuckDuckGo News Search)
    try:
        query = f"{ticker} stock finance market news"
        with DDGS() as ddgs:
            # We use .news() which is specialized for recent articles
            for r in ddgs.news(query, max_results=max_results):
                title = r.get('title', 'No Title')
                print("- ",title)
                summary = r.get('body', '')
                source = r.get('source', r.get('url', 'Unknown'))
                results.append(f"Title: {title}\nSummary: {summary}\nSource: {source}\n---")
    except Exception as e:
        print(f"  -> DDGS search encountered an issue: {e}. Switching to fallback tool...")

    # Attempt 2: Fallback Tool (Yahoo Finance Built-in News)
    if not results:
        print("  -> Attempting to fetch news from Yahoo Finance fallback...")
        try:
            stock = yf.Ticker(ticker)
            yf_news = stock.news[:max_results]
            for r in yf_news:
                title = r.get('title', 'No Title')
                # yfinance news sometimes stores summaries inside a 'content' dict
                summary = r.get('summary', r.get('content', {}).get('summary', 'No summary available.'))
                publisher = r.get('publisher', 'Yahoo Finance')
                results.append(f"Title: {title}\nSummary: {summary}\nSource: {publisher}\n---")
        except Exception as e:
            return f"Error: All news tools failed to retrieve data. Reason: {str(e)}"

    # Final Guardrail: If both tools found 0 articles
    if not results:
        return "No recent news articles could be found for this ticker."

    return "\n".join(results)


def run_news_agent(ticker: str) -> str:
    """Agent 1: Analyzes market sentiment and recent news with guardrails."""
    print(f"[Agent 1] Fetching live news for {ticker}...")
    raw_news = get_recent_news(ticker)
    
    # Guardrail: Prevent wasting an API call if no data was gathered
    if "No recent news articles could be found" in raw_news or "Error:" in raw_news:
        return f"Agent 1 Analysis Aborted: {raw_news}"
    
    prompt = f"""
    You are an expert Financial News & Sentiment Analyst.
    Analyze the following recent news headlines for ticker symbol: {ticker}.
    
    Raw News Data:
    {raw_news}
    
    Task:
    1. Summarize the overarching market sentiment (Bullish, Bearish, or Neutral).
    2. Identify the top 2-3 key catalysts or risks mentioned in the news.
    3. Keep your analysis concise, objective, and factual.
    """
    
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text


In [ ]:
def get_fundamental_data(ticker: str) -> str:
    """Pulls quantitative financial metrics using yfinance."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        metrics = {
            "Company Name": info.get("longName", ticker),
            "Current Price": info.get("currentPrice", "N/A"),
            "Market Cap": info.get("marketCap", "N/A"),
            "Trailing P/E": info.get("trailingPE", "N/A"),
            "Forward P/E": info.get("forwardPE", "N/A"),
            "52 Week High": info.get("fiftyTwoWeekHigh", "N/A"),
            "52 Week Low": info.get("fiftyTwoWeekLow", "N/A"),
            "50 Day Moving Average": info.get("fiftyDayAverage", "N/A"),
            "200 Day Moving Average": info.get("twoHundredDayAverage", "N/A"),
            "Revenue Growth (YoY)": info.get("revenueGrowth", "N/A"),
            "Analyst Recommendation": info.get("recommendationKey", "N/A").upper(),
            "Target Mean Price": info.get("targetMeanPrice", "N/A")
        }
        return json.dumps(metrics, indent=2)
    except Exception as e:
        return f"Could not retrieve fundamental data: {str(e)}"

def run_quant_agent(ticker: str) -> str:
    """Agent 2: Evaluates quantitative metrics and valuation."""
    print(f"[Agent 2] Extracting fundamental metrics for {ticker}...")
    raw_metrics = get_fundamental_data(ticker)
    
    prompt = f"""
    You are an expert Quantitative Financial Analyst.
    Evaluate the following valuation metrics and technical indicators for: {ticker}.
    
    Raw Financial Data:
    {raw_metrics}
    
    Task:
    1. Evaluate the valuation (e.g., is the P/E ratio overvalued or undervalued relative to historical norms?).
    2. Assess the price trend based on the 50-day and 200-day moving averages.
    3. Provide a clear assessment of the company's fundamental strength.
    """
    
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1)
    )
    return response.text



In [ ]:
def run_portfolio_manager_agent(ticker: str, news_analysis: str, quant_analysis: str) -> str:
    """Agent 3: Synthesizes research and issues an investment recommendation with crash-protection."""
    print(f"[Agent 3] Synthesizing reports and formulating investment thesis for {ticker}...")
    
    prompt = f"""
    You are the Chief Investment Officer and Lead Portfolio Manager at an elite hedge fund.
    You have commissioned two independent research reports on ticker symbol: {ticker}.
    
    === REPORT 1: NEWS & MARKET SENTIMENT ===
    {news_analysis}
    
    === REPORT 2: QUANTITATIVE & FUNDAMENTAL VALUATION ===
    {quant_analysis}
    
    Task:
    Weigh the bullish signals against the bearish risks. Provide a comprehensive investment memo structured EXACTLY with the following Markdown headings:
    
    # Investment Thesis for {ticker}
    ## Executive Summary
    ## Key Bullish Arguments
    ## Key Risks & Bearish Arguments
    ## Final Recommendation: [BUY / HOLD / SELL]
    *(State clearly whether to Buy, Hold, or Sell and explain your time horizon and reasoning).*
    """
    
    # Use new safe wrapper with fallback protection
    return safe_generate_content(
        prompt=prompt, 
        primary_model="gemini-3.5-flash", 
        fallback_model="gemini-3.1-flash-lite", 
        max_retries=3
    )

In [ ]:
def analyze_stock_pipeline(ticker: str):
    """Orchestrates the 3-agent workflow."""
    ticker = ticker.upper().strip()
    print(f"=== Starting Multi-Agent Analysis for {ticker} ===")\
    
    # Step 1: Run Researchers in parallel/sequence
    news_report = run_news_agent(ticker)
    quant_report = run_quant_agent(ticker)
    
    # Step 2: Run Lead Manager with research inputs
    final_memo = run_portfolio_manager_agent(ticker, news_report, quant_report)
    
    print("\n" + "="*50)
    print("=== FINAL INVESTMENT MEMO ===")
    print("="*50 + "\n")
    print(final_memo)
    return final_memo

# Execute your full system
final_output = analyze_stock_pipeline("NVDA")